## Reading from the .fdf

In [5]:
# --- Build & Display structure from the *latest* iteration of MTJ.out ---
# Now also parses %block DM.InitSpin and embeds per-atom initial magnetic moments
# Prints all numbers with 8 decimal places.

from pathlib import Path
import re
import numpy as np
import pandas as pd
from pymatgen.core import Lattice, Structure

OUT_PATH = Path("MTJ.out")   # adjust path if needed

# ------------------ helpers ------------------
FLOAT = r"[+-]?(?:\d+\.\d*|\.\d+|\d+)(?:[Ee][+-]?\d+)?"

def _find_all_indices(lines, pattern):
    prog = re.compile(pattern)
    return [i for i, ln in enumerate(lines) if prog.search(ln)]

def _extract_last_outcell_block(lines):
    idxs = _find_all_indices(lines, r"^\s*outcell:\s*Unit cell vectors")
    if not idxs:
        raise RuntimeError("Couldn't find 'outcell: Unit cell vectors' block.")
    start = idxs[-1]
    vecs = []
    for ln in lines[start+1:start+12]:
        nums = re.findall(FLOAT, ln)
        if len(nums) >= 3:
            vecs.append(list(map(float, nums[:3])))
            if len(vecs) == 3:
                break
    if len(vecs) != 3:
        raise RuntimeError("Failed to parse 3×3 lattice from the last outcell block.")
    return np.array(vecs, dtype=float)

def _extract_last_outcoor_block(lines):
    idxs = _find_all_indices(lines, r"^\s*outcoor:\s*Atomic coordinates")
    frac_idxs = _find_all_indices(lines, r"^\s*outcoor:\s*Fractional coordinates")
    all_idxs = sorted(idxs + frac_idxs)
    if not all_idxs:
        raise RuntimeError("Couldn't find 'outcoor: Atomic/Fractional coordinates' block.")
    start = all_idxs[-1]
    header = lines[start].lower()
    is_fractional = any(k in header for k in ("fractional", "direct", "scaled"))

    rows = []
    for ln in lines[start+1:]:
        s = ln.strip()
        if not s:
            if rows:
                break
            else:
                continue
        if re.match(r"^\s*out\w+:", s):
            break
        parts = s.split()
        if len(parts) >= 4 and parts[0].isdigit():
            parts = parts[1:4]
        elif len(parts) >= 3:
            parts = parts[:3]
        else:
            if rows:
                break
            else:
                continue
        try:
            rows.append([float(x) for x in parts])
        except Exception:
            if rows:
                break
            else:
                continue
    if not rows:
        raise RuntimeError("Parsed zero coordinates from the last outcoor block.")
    return ("frac" if is_fractional else "cart"), np.array(rows, dtype=float)

def _read_block(lines, start_idx):
    out = []
    for j in range(start_idx+1, len(lines)):
        ln = lines[j].rstrip("\n")
        if ln.strip().lower().startswith("%endblock"):
            break
        out.append(ln)
    return out

def _species_map_from_label_block(block_lines):
    mapping = {}
    for ln in block_lines:
        parts = ln.split()
        if len(parts) >= 3 and parts[2].isalpha():
            mapping[int(parts[0])] = parts[2]
        elif len(parts) >= 2 and parts[1].isalpha():
            mapping[int(parts[0])] = parts[1]
    return mapping

def _species_order_from_echoed_atomic_block(lines):
    begin_idxs = _find_all_indices(lines, r"^\s*%block\s+AtomicCoordinates(?:AndAtomicSpecies|AndSpecies|)?\s*$")
    if not begin_idxs:
        return None
    begin = begin_idxs[-1]
    end = None
    for j in range(begin+1, min(len(lines), begin+2000)):
        if lines[j].strip().lower().startswith("%endblock"):
            end = j; break
    atomic_block = lines[begin+1:end] if end else lines[begin+1:begin+1000]

    # ChemicalSpeciesLabel map
    spec_begin_idxs = _find_all_indices(lines, r"^\s*%block\s+ChemicalSpeciesLabel\s*$")
    species_map = {}
    if spec_begin_idxs:
        s_begin = spec_begin_idxs[-1]
        s_end = None
        for j in range(s_begin+1, min(len(lines), s_begin+500)):
            if lines[j].strip().lower().startswith("%endblock"):
                s_end = j; break
        spec_block = lines[s_begin+1:s_end] if s_end else lines[s_begin+1:s_begin+200]
        species_map = _species_map_from_label_block(spec_block)

    species = []
    for ln in atomic_block:
        parts = ln.split()
        if len(parts) < 4:
            continue
        tag = parts[3]
        sym = species_map.get(int(tag), "X") if tag.isdigit() else tag
        species.append(sym)
    return species

def _frac_to_cart(frac, lattice):
    return frac @ lattice

# --- Parse %block DM.InitSpin (index + value OR index + vector) ---
def _parse_dm_init_spin(lines):
    """
    Returns:
      - mag_list: list of scalars (μB) OR list of [mx,my,mz] (μB), length = number of atoms found in block
      - is_vector: True if 3 components/atom were found in any line
      - by_index: dict {atom_index(int, 1-based): value or [mx,my,mz]}
    If not found, returns (None, False, {}).
    """
    begin_idxs = _find_all_indices(lines, r"^\s*%block\s+DM\.?InitSpin\s*$")
    if not begin_idxs:
        return None, False, {}
    begin = begin_idxs[-1]
    end = None
    for j in range(begin+1, min(len(lines), begin+2000)):
        if lines[j].strip().lower().startswith("%endblock"):
            end = j; break
    block = lines[begin+1:end] if end else lines[begin+1:begin+2000]

    by_index = {}
    is_vec = False
    for ln in block:
        s = ln.strip()
        if not s or s.startswith("#"):
            continue
        nums = re.findall(FLOAT, s)
        if len(nums) < 2:
            continue
        idx = int(float(nums[0]))
        # If 3+ after index → treat as vector; else single scalar
        if len(nums) >= 4:
            val = [float(nums[1]), float(nums[2]), float(nums[3])]
            is_vec = True
        else:
            val = float(nums[1])
        by_index[idx] = val

    if not by_index:
        return None, False, {}

    # Produce ordered list by increasing index (1..max_idx)
    max_idx = max(by_index)
    mags = []
    for i in range(1, max_idx+1):
        if i in by_index:
            mags.append(by_index[i])
        else:
            mags.append([np.nan, np.nan, np.nan] if is_vec else np.nan)

    # Normalize if vector mode detected anywhere: pad scalars to vectors [m,0,0]
    if is_vec:
        mags = [m if isinstance(m, (list, tuple)) else [m, 0.0, 0.0] for m in mags]

    return mags, is_vec, by_index

# ------------ parse latest iteration ------------
lines = OUT_PATH.read_text(errors="ignore").splitlines()
lattice = _extract_last_outcell_block(lines)
kind, coords = _extract_last_outcoor_block(lines)
species = _species_order_from_echoed_atomic_block(lines)
if species is None or len(species) != len(coords):
    species = (species or [])[:len(coords)] + ["X"] * max(0, len(coords) - (len(species) or 0))
cart = _frac_to_cart(coords, lattice) if kind == "frac" else coords

# 1) Build structure FIRST (no magmom yet)
pmg_latt = Lattice(lattice)
structure = Structure(pmg_latt, species, cart, coords_are_cartesian=True)

# 2) Parse DM.InitSpin and ADD site property *after* creating the structure
mag_list, mag_is_vec, mag_by_index = _parse_dm_init_spin(lines)

if mag_list is not None:
    n_sites = len(structure)
    if mag_is_vec:
        mag_init = []
        for i in range(1, n_sites+1):
            val = mag_by_index.get(i, [np.nan, np.nan, np.nan])
            if not isinstance(val, (list, tuple)) or len(val) < 3:
                val = [float(val), 0.0, 0.0]
            mag_init.append(val)
    else:
        mag_init = []
        for i in range(1, n_sites+1):
            val = mag_by_index.get(i, np.nan)
            if isinstance(val, (list, tuple)):
                val = float(val[2]) if len(val) >= 3 else float(val[0])
            mag_init.append(float(val) if val is not np.nan else np.nan)

    # attach as "magmom" (your visualization code expects this name)
    structure.add_site_property("magmom", mag_init)

# ------------------ build structure ------------------
pmg_latt = Lattice(lattice)
structure = Structure(pmg_latt, species, cart, coords_are_cartesian=True, site_properties={"magmom": mag_init})

# Attach initial magmoms as site property "magmom_init"
if mag_list is not None:
    # Align by atom index (1-based). If DM.InitSpin covers fewer atoms than structure, pad with NaN.
    n_sites = len(structure)
    if mag_is_vec:
        # Build length n_sites list of 3-vectors
        mag_init = []
        for i in range(1, n_sites+1):
            if i in mag_by_index:
                val = mag_by_index[i]
                val = val if isinstance(val, (list, tuple)) and len(val) >= 3 else [float(val), 0.0, 0.0]
            else:
                val = [np.nan, np.nan, np.nan]
            mag_init.append(val)
        structure.add_site_property("magmom", mag_init)
    else:
        mag_init = []
        for i in range(1, n_sites+1):
            val = mag_by_index.get(i, np.nan)
            # if someone entered a vector accidentally, reduce to magnitude/sign along z
            if isinstance(val, (list, tuple)):
                val = float(val[2]) if len(val) >= 3 else float(val[0])
            mag_init.append(float(val) if val is not np.nan else np.nan)
        structure.add_site_property("magmom", mag_init)

# ------------------ display structure ------------------
display(structure)  # rich pprint in Jupyter; site properties listed under each site

# ------------------ print lattice vectors (8 decimals) ------------------
print("\nLATTICE VECTORS (Å) — latest iteration")
for i, v in enumerate(lattice, start=1):
    print(f"  v{i}: {v[0]:.8f}  {v[1]:.8f}  {v[2]:.8f}")
a_len, b_len, c_len = (np.linalg.norm(lattice[i]) for i in range(3))
print(f"a,b,c lengths (Å): {a_len:.8f}, {b_len:.8f}, {c_len:.8f}")

# ------------------ print coordinates (+ DM.InitSpin) with 8 decimals ------------------
data = {
    "index": np.arange(1, len(cart)+1, dtype=int),
    "species": species,
    "x (Å)": cart[:,0],
    "y (Å)": cart[:,1],
    "z (Å)": cart[:,2],
}

if "magmom" in structure.site_properties:
    if mag_is_vec:
        mx, my, mz = [], [], []
        for m in structure.site_properties["magmom"]:
            if isinstance(m, (list, tuple)) and len(m) >= 3:
                mx.append(m[0]); my.append(m[1]); mz.append(m[2])
            else:
                mx.append(np.nan); my.append(np.nan); mz.append(np.nan)
        data["mx_init (μB)"] = np.array(mx, dtype=float)
        data["my_init (μB)"] = np.array(my, dtype=float)
        data["mz_init (μB)"] = np.array(mz, dtype=float)
    else:
        data["m_init (μB)"] = np.array(structure.site_properties["magmom"], dtype=float)

df = pd.DataFrame(data)

# enforce 8-decimal printing
with pd.option_context("display.max_rows", None):
    pd.set_option("display.float_format", lambda x: f"{x:.8f}")
    print("\nCARTESIAN COORDINATES (Å) + DM.InitSpin — latest iteration")
    display(df)
    # reset float format to pandas default for safety in later cells
    pd.reset_option("display.float_format")


Structure Summary
Lattice
    abc : 4.045 4.045 50.522584
 angles : 90.0 90.0 90.0
 volume : 826.6517824726
      A : np.float64(4.045) np.float64(0.0) np.float64(0.0)
      B : np.float64(0.0) np.float64(4.045) np.float64(0.0)
      C : np.float64(0.0) np.float64(0.0) np.float64(50.522584)
    pbc : True True True
PeriodicSite: Fe (3.034, 3.034, 0.6687) [0.75, 0.75, 0.01323]
PeriodicSite: Fe (1.011, 1.011, 0.6702) [0.25, 0.25, 0.01327]
PeriodicSite: Fe (3.034, 1.011, 2.09) [0.75, 0.25, 0.04136]
PeriodicSite: Fe (1.011, 3.034, 2.09) [0.25, 0.75, 0.04136]
PeriodicSite: Fe (3.034, 3.034, 3.492) [0.75, 0.75, 0.06913]
PeriodicSite: Fe (1.011, 1.011, 3.495) [0.25, 0.25, 0.06919]
PeriodicSite: Fe (3.034, 1.011, 4.913) [0.75, 0.25, 0.09723]
PeriodicSite: Fe (1.011, 3.034, 4.913) [0.25, 0.75, 0.09723]
PeriodicSite: Fe (3.034, 3.034, 6.309) [0.75, 0.75, 0.1249]
PeriodicSite: Fe (1.011, 1.011, 6.31) [0.25, 0.25, 0.1249]
PeriodicSite: Fe (3.034, 1.011, 7.71) [0.75, 0.25, 0.1526]
PeriodicSite: Fe 


LATTICE VECTORS (Å) — latest iteration
  v1: 4.04500000  0.00000000  0.00000000
  v2: 0.00000000  4.04500000  0.00000000
  v3: 0.00000000  0.00000000  50.52258400
a,b,c lengths (Å): 4.04500000, 4.04500000, 50.52258400

CARTESIAN COORDINATES (Å) + DM.InitSpin — latest iteration


,index,species,x (Å),y (Å),z (Å),m_init (μB)
0,1,Fe,3.03375000,3.03375000,0.66865356,2.20000000
1,2,Fe,1.01125000,1.01125000,0.67019713,2.20000000
2,3,Fe,3.03375000,1.01125000,2.08979412,2.20000000
3,4,Fe,1.01125000,3.03375000,2.08979419,2.20000000
4,5,Fe,3.03375000,3.03375000,3.49238060,2.20000000
5,6,Fe,1.01125000,1.01125000,3.49545329,2.20000000
6,7,Fe,3.03375000,1.01125000,4.91256294,2.20000000
7,8,Fe,1.01125000,3.03375000,4.91256298,2.20000000
8,9,Fe,3.03375000,3.03375000,6.30946064,2.20000000
9,10,Fe,1.01125000,1.01125000,6.30969593,2.20000000


In [6]:
# Extract species, coordinates, and magnetic moments from the structure
Dev_species_L = [site.species_string for site in structure]
Dev_coords_L = [site.coords.tolist() for site in structure]
Dev_magmoms_L = structure.site_properties['magmom']

# Now you can use these lists to create a new structure or for other purposes
print("Species:", Dev_species_L)
print("Coordinates:", Dev_coords_L)
print("Magnetic moments:", Dev_magmoms_L)


Species: ['Fe', 'Fe', 'Fe', 'Fe', 'Fe', 'Fe', 'Fe', 'Fe', 'Fe', 'Fe', 'Fe', 'Fe', 'Fe', 'Fe', 'Fe', 'Fe', 'Fe', 'Fe', 'Fe', 'Fe', 'Fe', 'Fe', 'Fe', 'Fe', 'Mg', 'Mg', 'Mg', 'Mg', 'O', 'O', 'O', 'O', 'Mg', 'Mg', 'Mg', 'Mg', 'O', 'O', 'O', 'O', 'Ge', 'Mn', 'Mn', 'Mn', 'Mn', 'Ge', 'Mn', 'Mn', 'Ge', 'Mn', 'Mn', 'Mn', 'Ge', 'Mn', 'Mn', 'Mn', 'Ge', 'Mn', 'Mn', 'Mn', 'Ge', 'Mn', 'Mn', 'Mn', 'Ge', 'Mn', 'Mn', 'Mn', 'Mn', 'Ge', 'Mn', 'Mn']
Coordinates: [[3.03375, 3.03375, 0.66865356], [1.01125, 1.01125, 0.6701971300000001], [3.03375, 1.01125, 2.0897941200000005], [1.01125, 3.03375, 2.0897941900000006], [3.03375, 3.03375, 3.4923806], [1.01125, 1.01125, 3.49545329], [3.03375, 1.01125, 4.912562940000001], [1.01125, 3.03375, 4.91256298], [3.03375, 3.03375, 6.309460640000001], [1.01125, 1.01125, 6.309695930000001], [3.03375, 1.01125, 7.71009392], [1.01125, 3.03375, 7.7100939], [3.03375, 3.03375, 9.10101056], [1.01125, 1.01125, 9.100511910000002], [3.03375, 1.01125, 10.50255948], [1.01125, 3.03375, 10

In [7]:
# Create a dictionary with your structure data
Dev = {
    'Dev_species': [site.species_string for site in structure],
    'Dev_coords': [site.coords.tolist() for site in structure],
    'Dev_magmoms': structure.site_properties['magmom']
}

# Save using pickle (recommended for reliability)
import pickle
with open('Dev.pkl', 'wb') as f:
    pickle.dump(Dev, f)

# Alternative quick storage with %store magic
%store Dev

Stored 'Dev' (dict)
